# Phase 5 · TF-IDF variants for the ensemble (CPU, High-RAM runtime)
The ensemble notebook found that mixing a TF-IDF + linear model into the neural runs adds macro-F1. This notebook trains several versions of that model (feature sets x regularisation x class weighting), scores each alone and mixed with the neural ensemble (both with honest cross-fitted thresholds on dev), saves every version's probabilities, and copies the best mixed one to `runs/phase5_lsi/tfidf/probs.npz`, which is what the ensemble notebook loads. Every fit prints its time; trim `FEATURES` / `MODELS` if you are short of time.

In [ ]:
# 1 · Settings
from pathlib import Path
SEED = 17
COURSE_ROOT = Path('/content/drive/MyDrive/legal-llm-course'); PHASE_DIR = COURSE_ROOT/'Phase_5_LSI'; RUNS_ROOT = COURSE_ROOT/'runs/phase5_lsi'
BUNDLE_ZIP  = COURSE_ROOT/'phase2_lner_multimethod_bundle.zip'
RR_LIB, CJPE_LIB, LSI_LIB = COURSE_ROOT/'rr_lib.py', COURSE_ROOT/'cjpe_lib.py', COURSE_ROOT/'lsi_lib.py'
# feature sets: name -> (ngram range, max features, min document frequency, characters kept per document)
FEATURES = {'w12': ((1, 2), 200000, 3, 60000), 'w13': ((1, 3), 400000, 3, 120000)}
# linear models on top: name -> (LinearSVC C, class weighting)   'balanced' up-weights the rare statutes, which is what macro-F1 rewards
MODELS = {'svm0.1': (0.1, None), 'svm0.5': (0.5, None), 'svm0.1bal': (0.1, 'balanced'), 'svm0.5bal': (0.5, 'balanced')}
MIX_WEIGHT, SHRINK_M = 0.5, 30       # how each version is judged when mixed with the neural ensemble (the best setting found in the ensemble notebook)
CANDIDATES = [RUNS_ROOT/f'NEW-{m}/full/new-{n}-full-00{i}' for i in (1, 2) for m, n in (('GQA', 'gqa'), ('ENC', 'enc'), ('HYB', 'hyb'))]

In [ ]:
# 2 · Drive, helper bundle, tokenizer, device
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, zipfile, hashlib, json, subprocess, importlib.util, random
LOCAL = Path('/content/phase3-rr')
if LOCAL.exists(): shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as z: z.extractall(LOCAL)
shutil.copy(RR_LIB, LOCAL/'rr_lib.py'); shutil.copy(CJPE_LIB, LOCAL/'cjpe_lib.py'); shutil.copy(LSI_LIB, LOCAL/'lsi_lib.py')
sys.path.insert(0, str(LOCAL))
need = [p for p in ('tokenizers','huggingface_hub','scikit-learn') if importlib.util.find_spec(p.replace('scikit-learn','sklearn')) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])
import torch, rr_lib as R, cjpe_lib as J, lsi_lib as LSI
from tokenizers import Tokenizer
from training_methods import resolve_snapshot_checkpoint
TOKENIZER_SHA256 = '906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab'
tp = LOCAL/'tokenizer/tokenizer.json'
assert hashlib.sha256(tp.read_bytes()).hexdigest() == TOKENIZER_SHA256, 'tokenizer differs from Phase 1/2'
TOK = Tokenizer.from_file(str(tp))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); torch.manual_seed(SEED)
import numpy as np
print('device', device, torch.cuda.get_device_name(0) if device.type=='cuda' else '')

In [ ]:
# 3 · Data, and the neural ensemble's dev probabilities (the thing each TF-IDF version is mixed with)
import time, pandas as pd
tr_docs = LSI.load_split(PHASE_DIR/'Train/train.json', dedup=True); dev_docs = LSI.load_split(PHASE_DIR/'Train/dev.json', dedup=True); test_docs = LSI.load_split(PHASE_DIR/'Test/test.json', keep_labels=False)
ytr, Y = LSI.multi_hot(tr_docs), LSI.multi_hot(dev_docs); dev_ids = [d['id'] for d in dev_docs]; test_ids = [d['id'] for d in test_docs]
RUN_DIRS = [r for r in CANDIDATES if (r/'dev_probs.npy').exists()]
print('neural runs found:', [r.name for r in RUN_DIRS])
P_NN = np.mean([np.load(r/'dev_probs.npy') for r in RUN_DIRS], 0) if RUN_DIRS else None
def cross_fit_mix(P_tf, wt=MIX_WEIGHT, m=SHRINK_M, seeds=5):
    # mean macro-F1 over repeated 2-fold splits of dev: thresholds (shrunk toward the global one) tuned on one half, scored on the other
    P = P_tf if P_NN is None else (1 - wt) * P_NN + wt * P_tf; out = []
    for sd in range(seeds):
        halves = np.array_split(np.random.RandomState(sd).permutation(len(Y)), 2)
        for a, b in ((0, 1), (1, 0)):
            tr, te = halves[a], halves[b]; f = LSI._f1_by_threshold(Y[tr], P[tr]); n = Y[tr].sum(0)
            g = LSI.GRID[int(np.argmax(f.mean(1)))]; t = LSI.GRID[np.argmax(f, 0)]; lam = n / (n + m)
            out.append(LSI.scores(Y[te], LSI.decide(P[te], lam * t + (1 - lam) * g, True))['macro_f1'])
    return float(np.mean(out))

In [ ]:
# 4 · Train every version (the slow cell: vectorising the 42,000 training documents dominates, then each fit takes a few minutes)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
VAR = RUNS_ROOT/'tfidf/variants'; VAR.mkdir(parents=True, exist_ok=True)
sig = lambda S: (1 / (1 + np.exp(-4 * S))).astype(np.float32)        # squash scores to (0,1), as in the baseline notebook
rows = []
for fname, (ngram, maxf, mindf, chars) in FEATURES.items():
    t0 = time.time(); cut = lambda ds: [d['text'][:chars] for d in ds]
    vec = TfidfVectorizer(ngram_range=ngram, min_df=mindf, max_features=maxf, sublinear_tf=True, dtype=np.float32)
    A = vec.fit_transform(cut(tr_docs)); B = vec.transform(cut(dev_docs)); T = vec.transform(cut(test_docs))
    print(f'{fname}: matrix {A.shape} in {time.time() - t0:.0f} s', flush=True)
    for mname, (C_, cw) in MODELS.items():
        name = f'{fname}_{mname}'; f = VAR/f'{name}.npz'
        if f.exists() and list(np.load(f)['dev_ids']) == dev_ids:
            z = np.load(f); P_dev, P_test = z['dev'], z['test']; how = 'cached'
        else:
            t1 = time.time(); clf = OneVsRestClassifier(LinearSVC(C=C_, class_weight=cw), n_jobs=-1).fit(A, ytr)
            P_dev, P_test = sig(clf.decision_function(B)), sig(clf.decision_function(T)); how = f'{time.time() - t1:.0f} s'
            np.savez(f, dev=P_dev, test=P_test, dev_ids=np.array(dev_ids), test_ids=np.array(test_ids), name=name)
        alone = LSI.evaluate_probs(Y, P_dev)['per_label_cross_fit+min_one']['macro_f1']; mixed = cross_fit_mix(P_dev)
        rows.append({'version': name, 'alone (per-label cross-fit)': round(alone, 4), f'mixed with neural ensemble (w={MIX_WEIGHT})': round(mixed, 4), 'fit': how}); print(rows[-1], flush=True)

In [ ]:
# 5 · Pick the version that helps the ensemble most and hand it to the ensemble notebook
col = f'mixed with neural ensemble (w={MIX_WEIGHT})'
tab = pd.DataFrame(rows).sort_values(col, ascending=False); display(tab)
best = tab.iloc[0]['version']; z = np.load(VAR/f'{best}.npz')
np.savez(RUNS_ROOT/'tfidf/probs.npz', dev=z['dev'], test=z['test'], dev_ids=z['dev_ids'], test_ids=z['test_ids'], name=best)
print('best version:', best, '-> copied to', RUNS_ROOT/'tfidf/probs.npz', '(the ensemble notebook loads it from there; rerun its cells 5-7)')